# 11주차 : 분석용 표와 전처리 로그
**분석 질문:** 구독료를 포함한 대학생 소비 목록의 물가지수는 2017년 1월부터 2026년 9월까지 전체 소비자물가지수와 최저시급보다 더 빠르게 올랐나?

**원자료**
(GitHub [skyseoker/Economic-Data-Analytics](https://github.com/skyseoker/Economic-Data-Analytics))
1. `품목별_소비자물가지수.csv` · KOSIS 품목별 소비자물가지수(품목성질별: 2020=100), 전국, 총지수 + 25개 품목, 2017.01~2026.10 월별 (조회 2026-11-16)
2. `최저임금.xlsx` · e-나라지표 1492 최저임금 일반현황, 2017~2026 연별 (조회 2026-11-09)

In [1]:
#1. 파일을 GitHub에서 불러오기
import os, re, unicodedata
from urllib.parse import quote
import pandas as pd

BASE = "https://raw.githubusercontent.com/skyseoker/Economic-Data-Analytics/main/"
CPI_FILE = "품목별_소비자물가지수.csv"
MW_FILE  = "최저임금.xlsx"
START, END = "2017-01-01", "2026-09-01"   # 분석 기간 설정하기

def read_data(reader, name, **kw):
    """같은 폴더에 파일이 있으면 그것을, 없으면 GitHub에서 읽는다.
    맥에서 올린 한글 파일명(NFD)도 찾을 수 있게 두 형태를 모두 시도한다."""
    if os.path.exists(name):
        print("로컬 파일:", name)
        return reader(name, **kw)
    err = None
    for form in ("NFC", "NFD"):
        url = BASE + quote(unicodedata.normalize(form, name))
        try:
            data = reader(url, **kw)
            print("GitHub에서 읽음:", name, f"({form})")
            return data
        except Exception as e:
            err = e
    raise FileNotFoundError(f"{name}을(를) 찾지 못함") from err

In [2]:
# 2. 품목별 소비자 물가 지수 읽기
raw = read_data(pd.read_csv, CPI_FILE, encoding="utf-8-sig", na_values=["-", ""])
print("처리 전:", raw.shape)
raw.iloc[:, :6].head()

GitHub에서 읽음: 품목별_소비자물가지수.csv (NFD)
처리 전: (26, 120)


,시도별,품목별,2017.01,2017.02,2017.03,2017.04
0,전국,총지수,97.366,97.632,97.565,97.442
1,전국,월세,100.552,100.622,100.612,100.582
2,전국,도시철도료,99.009,99.009,99.009,99.009
3,전국,시내버스료,95.433,95.433,95.433,95.433
4,전국,휴대전화료,108.890,108.890,108.890,108.890


In [3]:
# 3-1. 시도별 열 지우기 : 관계 없는 기준 제거
before = raw.shape
cpi = raw.drop(columns="시도별")
print("처리 전", before, "처리 후", cpi.shape)

처리 전 (26, 120) 처리 후 (26, 119)


In [4]:
# 3-2. 결측치·빈칸 점검 → 있으면 삭제
time_cols = [c for c in cpi.columns if re.fullmatch(r"\d{4}\.\d{2}", str(c))]

# ① 결측치 칸 찾기
na_cells = int(cpi.isna().sum().sum())
# ② 빈 품목명 찾기
blank_name = int(cpi["품목별"].astype(str).str.strip().isin(["", "nan"]).sum())
# ③ 숫자가 아닌 값 남은 열 찾기
non_num = [c for c in time_cols if not pd.api.types.is_numeric_dtype(cpi[c])]

print("결측치 칸:", na_cells, "| 빈 품목명:", blank_name, "| 숫자가 아닌 값의 열:", len(non_num))
print("열별 결측이 있는 열:", cpi.columns[cpi.isna().any()].tolist())

# 결측이 하나라도 있는 행 삭제 (없으면 그대로)
before = cpi.shape
cpi = cpi.dropna().reset_index(drop=True)
print("처리 전", before, "처리 후", cpi.shape, "| 삭제한 행:", before[0] - cpi.shape[0])

결측치 칸: 0 | 빈 품목명: 0 | 숫자가 아닌 값의 열: 0
열별 결측이 있는 열: []
처리 전 (26, 119) 처리 후 (26, 119) | 삭제한 행: 0


In [5]:
# 4. 품목 분류하기
groups = {
    "비교 기준": ["총지수"],
    "교통비": ["시내버스료", "도시철도료"],
    "식비": ["김밥", "떡볶이", "치킨", "햄버거", "라면(외식)", "커피(외식)", "구내식당식사비"],
    "통신·구독비": ["온라인콘텐츠이용료", "휴대전화료", "인터넷이용료"],
    "주거비": ["월세", "학교기숙사비"],
    "문화생활비": ["PC방이용료", "영화관람료", "공연예술관람료", "관람시설이용료", "운동경기관람료"],
    "교육비": ["국공립대학교납입금", "사립대학교납입금", "전문대학납입금",
             "국공립대학원납입금", "사립대학원납입금", "시험응시료"],
}
item_to_group = {item: g for g, items in groups.items() for item in items}

cpi = cpi.drop(columns="분류", errors="ignore").copy()   # 이미 '분류' 열이 있으면 지우고 다시 만든다(셀을 여러 번 돌려도 안전)
cpi.insert(1, "분류", cpi["품목별"].map(item_to_group))

# 가나다순으로 정렬하기 (총지수는 맨 앞에 두기)
cpi["_기준"] = (cpi["품목별"] != "총지수")
cpi = (cpi.sort_values(["_기준", "분류", "품목별"])
          .drop(columns="_기준")
          .reset_index(drop=True))

print("분류 안 된 품목:", cpi.loc[cpi["분류"].isna(), "품목별"].tolist())
print("목록에만 있고 데이터에 없는 품목:", sorted(set(item_to_group) - set(cpi["품목별"])))
print(cpi["분류"].value_counts(sort=False).to_string())
print("모양:", cpi.shape)
cpi[["분류", "품목별", "2017.01", "2026.09"]]

분류 안 된 품목: []
목록에만 있고 데이터에 없는 품목: []
분류
비교 기준     1
교육비       6
교통비       2
문화생활비     5
식비        7
주거비       2
통신·구독비    3
모양: (26, 120)


,분류,품목별,2017.01,2026.09
0,비교 기준,총지수,97.366,120.43
1,교육비,국공립대학교납입금,102.222,101.67
2,교육비,국공립대학원납입금,98.624,107.95
3,교육비,사립대학교납입금,102.325,107.90
4,교육비,사립대학원납입금,98.241,109.10
5,교육비,시험응시료,94.619,116.55
6,교육비,전문대학납입금,100.246,108.68
7,교통비,도시철도료,99.009,120.08
8,교통비,시내버스료,95.433,116.96
9,문화생활비,PC방이용료,94.051,116.76


In [6]:
# 5. 중복·범위·이상치 점검하기
time_cols = [c for c in cpi.columns if re.fullmatch(r"\d{4}\.\d{2}", str(c))]

# 중복: 완전히 같은 행 또는 같은 품목이 두 번 들어간 행
print("완전히 같은 행:", int(cpi.duplicated().sum()),
      "| 품목명 중복:", int(cpi["품목별"].duplicated().sum()))

# 범위: 전체 지수의 최솟값·최댓값과 그 위치
vals = cpi.set_index("품목별")[time_cols]
lo_item, lo_t = vals.stack().idxmin()
hi_item, hi_t = vals.stack().idxmax()
print(f"지수 범위: 최솟값 {vals.min().min()} ({lo_item} {lo_t}) ~ 최댓값 {vals.max().max()} ({hi_item} {hi_t})")

# 이상치 후보 (플마 15%)
chg = vals.pct_change(axis=1) * 100
flag = chg.stack().dropna()
flag = flag[flag.abs() >= 15]

out = flag.rename("전월대비(%)").round(1).reset_index()
out.columns = ["품목별", "시점", "전월대비(%)"]
out["분류"] = out["품목별"].map(item_to_group)
out["전월 값"] = [vals.loc[i, time_cols[time_cols.index(t) - 1]] for i, t in zip(out["품목별"], out["시점"])]
out["이번 달 값"] = [vals.loc[i, t] for i, t in zip(out["품목별"], out["시점"])]

print("전월 대비 변화율 범위:", round(chg.min().min(), 1), "% ~", round(chg.max().max(), 1), "%")
print("±15% 이상인 칸:", len(out), "→ 삭제 0 (원인 확인 후 유지)")
out[["분류", "품목별", "시점", "전월 값", "이번 달 값", "전월대비(%)"]]

완전히 같은 행: 0 | 품목명 중복: 0
지수 범위: 최솟값 80.52 (휴대전화료 2025.08) ~ 최댓값 146.26 (김밥 2026.10)
전월 대비 변화율 범위: -25.2 % ~ 30.7 %
±15% 이상인 칸: 8 → 삭제 0 (원인 확인 후 유지)


,분류,품목별,시점,전월 값,이번 달 값,전월대비(%)
0,문화생활비,관람시설이용료,2020.05,109.39,81.85,-25.2
1,문화생활비,관람시설이용료,2020.06,81.85,106.96,30.7
2,문화생활비,관람시설이용료,2020.07,106.96,81.85,-23.5
3,문화생활비,관람시설이용료,2020.10,83.70,109.39,30.7
4,통신·구독비,휴대전화료,2020.10,101.72,80.58,-20.8
5,통신·구독비,휴대전화료,2020.11,80.58,99.43,23.4
6,통신·구독비,휴대전화료,2025.08,101.94,80.52,-21.0
7,통신·구독비,휴대전화료,2025.09,80.52,101.98,26.7


In [7]:
# 6. 최저임금 읽기
mw_raw = read_data(pd.read_excel, MW_FILE, header=None, dtype=str)
print("최저임금 원자료:", mw_raw.shape)

# 연도 줄(숫자 4자리만 있는 줄)과 '시간급 최저임금' 줄 찾기
label = mw_raw[0].astype(str).str.replace("\xa0", " ").str.strip()
is_year_row = mw_raw.iloc[:, 1:].apply(lambda r: r.astype(str).str.fullmatch(r"\d{4}").all(), axis=1)
years = mw_raw[is_year_row].iloc[0, 1:].astype(int)
wages = pd.to_numeric(mw_raw[label.str.startswith("시간급")].iloc[0, 1:].str.replace(",", ""))
wage_by_year = pd.Series(wages.values, index=years.values)
print("연도별 최저시급:", wage_by_year.to_dict())

# 달별로 펼치기: 연도별 -> 달별로 바꾸기
time_cols = [c for c in cpi.columns if re.fullmatch(r"\d{4}\.\d{2}", str(c))]
mw_row = {"분류": "비교 기준", "품목별": "최저시급(원)"}
mw_row.update({t: wage_by_year.get(int(t[:4])) for t in time_cols})

# 붙이기 (다시 실행해도 두 번 붙지 않게 먼저 지움) → 총지수 바로 아래에 둔다
before = cpi.shape
cpi = cpi[cpi["품목별"] != "최저시급(원)"]
cpi = pd.concat([cpi.iloc[:1], pd.DataFrame([mw_row]), cpi.iloc[1:]], ignore_index=True)

print("처리 전", before, "처리 후", cpi.shape)
print("최저시급 줄의 빈 달:", int(cpi.loc[cpi["품목별"] == "최저시급(원)", time_cols].isna().sum().sum()),
      "/", len(time_cols), "개월")
cpi.loc[:2, ["분류", "품목별", "2017.01", "2017.12", "2018.01", "2026.09", "2026.10"]]

GitHub에서 읽음: 최저임금.xlsx (NFD)
최저임금 원자료: (5, 11)
연도별 최저시급: {2017: 6470, 2018: 7530, 2019: 8350, 2020: 8590, 2021: 8720, 2022: 9160, 2023: 9620, 2024: 9860, 2025: 10030, 2026: 10320}
처리 전 (26, 120) 처리 후 (27, 120)
최저시급 줄의 빈 달: 0 / 118 개월


,분류,품목별,2017.01,2017.12,2018.01,2026.09,2026.10
0,비교 기준,총지수,97.366,97.698,98.106,120.43,120.73
1,비교 기준,최저시급(원),6470.000,6470.000,7530.000,10320.00,10320.00
2,교육비,국공립대학교납입금,102.222,101.833,101.833,101.67,101.67


In [8]:
# 7. 기간 필터: 2017.01 ~ 2026.09 열만 남기기 (lms에 업로드한 지난 과제들 날짜 기준)
id_cols   = ["분류", "품목별"]
time_cols = [c for c in cpi.columns if re.fullmatch(r"\d{4}\.\d{2}", str(c))]
start, end = START[:7].replace("-", "."), END[:7].replace("-", ".")   # "2017-01-01" → "2017.01"

keep = [t for t in time_cols if (t >= start) and (t <= end)]   # 'YYYY.MM' 문자는 크기 비교가 날짜 순서와 같음
drop = [t for t in time_cols if t not in keep]

before = cpi.shape
cpi = cpi[id_cols + keep]
print("처리 전", before, "처리 후", cpi.shape, "| 제외한 달:", drop)
print("경계값 확인: 첫 달", keep[0], "/ 끝 달", keep[-1], "/ 남은 달", len(keep), "개월")

처리 전 (27, 120) 처리 후 (27, 119) | 제외한 달: ['2026.10']
경계값 확인: 첫 달 2017.01 / 끝 달 2026.09 / 남은 달 117 개월


In [9]:
# 8. 저장 + 검산
cpi.to_csv("analysis_table.csv", index=False, encoding="utf-8-sig")

n_items = cpi["품목별"].ne("최저시급(원)").sum()
print(f"행 검산: 품목 {n_items} + 최저시급 1 = {n_items + 1} → 최종 {len(cpi)}행",
      "✔" if n_items + 1 == len(cpi) else "✘")
print(f"열(달) 검산: 원자료 {len(time_cols)}개월 - 제외 {len(drop)}개월 = {len(time_cols) - len(drop)} → 최종 {len(keep)}개월",
      "✔" if len(time_cols) - len(drop) == len(keep) else "✘")
print("분석용 표 shape:", cpi.shape, "| 결측:", int(cpi.isna().sum().sum()))
print("저장: analysis_table.csv (Colab 왼쪽 '파일' 탭에서 내려받기)")
cpi.iloc[:, [0, 1, 2, 3, -2, -1]].head()

행 검산: 품목 26 + 최저시급 1 = 27 → 최종 27행 ✔
열(달) 검산: 원자료 118개월 - 제외 1개월 = 117 → 최종 117개월 ✔
분석용 표 shape: (27, 119) | 결측: 0
저장: analysis_table.csv (Colab 왼쪽 '파일' 탭에서 내려받기)


,분류,품목별,2017.01,2017.02,2026.08,2026.09
0,비교 기준,총지수,97.366,97.632,120.05,120.43
1,비교 기준,최저시급(원),6470.000,6470.000,10320.00,10320.00
2,교육비,국공립대학교납입금,102.222,102.222,101.67,101.67
3,교육비,국공립대학원납입금,98.624,98.624,107.95,107.95
4,교육비,사립대학교납입금,102.325,102.325,107.92,107.90
